# Лабораторная работа 4. Введение в машинное обучение: knn, деревянные алгоритмы и ансамблирование. Реализация.

![](https://newapplift-production.s3.amazonaws.com/comfy/cms/files/files/000/001/201/original/machine-learning-robots-dilbert.gif)

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков Jupyter (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете так же должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам —  проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.


### Оценивание и штрафы
* Не копируйте классы между заданиями, объявите решающие модели один раз, а затем их инстанциируйте в каждой из ячеек
* Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи).
* Максимально допустимая оценка за работу — 14 балла. Дополнительных баллов не предусмотрено.
* Сдавать задание после указанного срока сдачи нельзя.
* Не оцениваются задания с удалёнными формулировками.
* Не оценивается лабораторная работа целиком, если она была выложена в открытый источник.


## Метрика качества

Обучение и оценка качества модели производятся на независимых множествах примеров. Как правило, имеющующиеся примеры разбивают на два подмножества: обучающее (train) и тестовое (test). Выбор пропорции разбиения — компромисс. Действительно, большой размер обучения ведет к более качественным алгоритмам, но бОльшему шуму при оценке модели на тесте. И наоборот, большой размер тестовой выборки ведет к менее шумной оценке качества, однако обученные модели получаются менее точными.

Многие модели классификации предсказывают оценку принадлежности положительному классу $\tilde{y}(x) \in R$ (например, вероятность принадлежности классу 1). После этого принимают решение о классе объекта путем сравнения оценки с некоторым порогом $\theta$:

$$y(x) = 
\begin{cases}
+1, &\text{если} \; \tilde{y}(x) \geq \theta \\
-1, &\text{если} \; \tilde{y}(x) < \theta
\end{cases}
$$

В этом случае можно рассматривать метрики, которые умеют работать с исходным ответом классификатора. В задании мы будем работать с метрикой AUC-ROC, которую в данном случае можно считать как долю неправильно упорядоченных пар объектов, отсортированных по возрастанию предсказанной оценки принадлежности классу 1 (более подробно можно узнать на следующих лекциях или, например, [здесь](https://github.com/esokolov/ml-course-msu/blob/master/ML15/lecture-notes/Sem05_metrics.pdf)). Детального понимания принципов работы метрики AUC-ROC для выполнения этой лабораторной не требуется. В sklearn данная метрика реализуется [следующей функцией](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

# Реализации алгоритмов классификации (14 баллов)

В данном задании вам предстоит сделать реализации уже пройденных алгоритмов. Критерии, по которым задания будут засчитываться следующие:

1. Правильность работы алгоритмов стоит сравнивать на [датасете с ирисами](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html), если не сказано обратного.
2. Алгоритм должен работать не хуже [бейзлайнов](https://datascience.stackexchange.com/questions/30912/what-does-baseline-mean-in-the-context-of-machine-learning) из sklearn при одинаковых гиперпараметрах - допустима ошибка 20% относительно бейзлайнов из sklearn. Т.е. если ваш алгоритм имеет `AUC-ROC 0.4`, а бейзлайн из sklearn `AUC-ROC 0.9`, то допустимым скором будет $0.9 - 0.2 \cdot 0.9 = 0.72$, ваш же скор меньше допустимого и равен $0.4$ - такой алгоритм зачесть нельзя. А если у вас `AUC-ROC 0.85`, то ваш скор выше допустимого - такой алгоритм зачесть можно. Пороги ошибки могут увеличиться в пользу студентов.
3. Код должен быть написан аккуратно, с понятными именами, разделением на функции и так далее. Чем выше неаккуратность кода, тем больше вероятность, что ~~его читать не будут~~ вы не получите комметариев, если вдруг у вас что-то не заведётся.
4. Никто не запрещает вам разбираться в готовых реализациях из интернета, мы даже это поощряем, не поощряем только бездумной копипасты. Если код будет содержать дополнительную функциональность, о которой не сказано в задании и вы не объясните, зачем вы её использовали и почему именно её - вам будет выставлено 0 баллов за весь раздел. Т.е. если вы ~~скопировали~~ реализовали decision tree и там используется специальная регуляризации в листьях дерева и вы не объясните зачем она нужна и почему она там, то вам за весь раздел с decision tree будет выставлено 0 баллов.

In [2]:
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True, as_frame=False)

Разделим выборку на train и test

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [4]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((100, 4), (50, 4), (100,), (50,))

## decision tree (13 баллов)

Интерфейс можете подсмотреть в [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier)

В данном разделе вам предстоит реализовать decision tree алгоритм для задачи классификации. Он должен поддерживать следующие параметры:
- максимальная глубина дерева (*max_depth*)



### plain decision tree (5.5 баллов)

Реализовать алгоритм решающего дерева, продолжив код с семинара. Теорию можно найти в ученике в главе [Решающие деревья](https://education.yandex.ru/handbook/ml/article/reshayushchiye-derevya)


**Задание 1. Наивная реализация decision tree (2б)**

Реализовать decision tree алгоритм с фиксированной глубиной дерева h=6. Если у вас алгоритм работает дольше, можете уменьшить глубину до 5 или 4.

В качестве алгоритма взять `Жадный алгоритм построения решающего дерева`. В качестве критерия ветвления выберите misclassification error. В данном задании вам не нужно делать никаких оптимизаций, ожидается наивное решение со сложностью $\mathcal{O}(hN^2D)$, где $h$ - высота дерева, $N$ - количество обучающих семплов и $D$ - размерность пространства фичей.


In [5]:
import numpy as np
np.random.seed(42)

In [6]:
from sklearn.base import BaseEstimator, ClassifierMixin


class TreeNode:
    def __init__(self, depth: int | None = None) -> None:
        self.depth = depth or 0
        self._answer = None
        self.feature_idx = None
        self.value = None
        self.left = None
        self.right = None

    @property
    def answer(self):
        assert self._answer is not None, "answer should be specified"
        return self._answer

    @answer.setter
    def answer(self, v: float):
        self._answer = v

    @property
    def is_leaf(self):
        assert (
            self.left is not None or self.right is None
        ), "all child should or shouldn't be"
        assert (
            self.left is None or self.right is not None
        ), "all child should or shouldn't be"
        return self.left is None and self.right is None

    def predict(self, x):
        if self.is_leaf:
            return self.answer
        
        assert (
            self.feature_idx is not None and self.value is not None
        ), "at prediction split values should be specified"

        if x[self.feature_idx] < self.value:
            return self.left.predict(x)
        else:
            return self.right.predict(x)


class MyCoolDecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        max_depth: int | None = None,
        min_samples_leaf: int | None = None,
        min_samples_split: int | None = None,
    ):
        self.max_depth = max_depth or 2
        self.min_samples_split = min_samples_split or 2
        self.min_samples_leaf = min_samples_leaf or 1
        self.root = TreeNode(depth=1)
        self._classes = None

    def _build_tree(self, node: TreeNode, X, y) -> TreeNode:
        
        if self._stop_criterion(node, X):
            node.answer = self._get_answer(X, y)
            return node

        _, optimal_border = self._get_split(X, y)
        if optimal_border is None:
            node.answer = self._get_answer(X, y)
            return node

        feature_idx, t = optimal_border 
        mask = (X[:, feature_idx] < t)
        node.feature_idx = feature_idx
        node.value = t

        X_l, y_l = X[mask], y[mask]
        X_r, y_r = X[~mask], y[~mask]

        left = TreeNode(node.depth + 1)
        right = TreeNode(node.depth + 1)
        node.left = self._build_tree(left, X_l, y_l)
        node.right = self._build_tree(right, X_r, y_r)

        return node


    def _stop_criterion(self, node: TreeNode, X) -> bool:
        if (
            node.depth >= self.max_depth
            or X.shape[0] < 2 * self.min_samples_leaf
            or X.shape[0] < self.min_samples_split
        ):
            return True
        else:
            return False
    

    def _get_split(self, X, y):
        min_loss = float("+inf")
        optimal_border = None

        for j in range(X.shape[1]):
            values = X[:, j]
           
            thresholds = (values[:-1] + values[1:]) / 2.0 # Используем значения порогов между значений фичей
            thresholds = thresholds[thresholds > 1e-12]
            for t in thresholds:
                loss = self._get_criterion(X, y, j, t)
                if loss < min_loss:
                    min_loss, optimal_border = loss, (j, t)

        return min_loss, optimal_border


    def _get_criterion(self, X, y, feature_idx, value):
        mask = (X[:, feature_idx] < value)
        X_l, y_l = X[mask], y[mask]
        X_r, y_r = X[~mask], y[~mask]

        return ((X_l.shape[0] / X.shape[0]) * self._H(X_l, y_l) + 
                (X_r.shape[0] / X.shape[0]) * self._H(X_r, y_r))


    def _H(self, X, y):
        if y.shape[0] == 0:
            return np.float32("+inf")
        cnts = np.unique_counts(y)
        k_count = cnts.counts.max()

        return 1. - k_count / X.shape[0]


    def _get_answer(self, X, y):
        result = []
        for k in self._classes:
            p_k = np.sum(y == k) / X.shape[0]
            result.append(p_k)

        return np.array(result)

    def _mse_loss(self, y):
        return np.var(y)

    def _mae_loss(self, y):
        return np.abs(y - np.median(y)).mean()

    def fit(self, X, y):
        self._classes = np.unique(y)
        self._build_tree(self.root, X, y)
        return self

    def predict_proba(self, X):
        scores = []
        for x in X:
            scores.append(self.root.predict(x))
        return np.array(scores)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

In [7]:
from sklearn.metrics import roc_auc_score


def compare_models(
        my_model: BaseEstimator, 
        baseline: BaseEstimator,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_test: np.ndarray,
        y_test: np.ndarray,
        is_verbose: bool = True):
    
    my_model.fit(X_train, y_train)
    baseline.fit(X_train, y_train)

    
    my_proba= my_model.predict_proba(X_test)
    baseline_proba = baseline.predict_proba(X_test)
    
    
    if np.unique(y_test).shape[0] == 2:
        my_proba = my_proba[:, 1]
        baseline_proba = baseline_proba[:, 1]
        my_roc_auc, baseline_roc_auc = roc_auc_score(y_test, my_proba), roc_auc_score(y_test, baseline_proba)
        
    else:
        my_roc_auc, baseline_roc_auc = roc_auc_score(y_test, my_proba, multi_class="ovr"), roc_auc_score(y_test, baseline_proba, multi_class="ovr")


    limit = baseline_roc_auc * 0.8
    if is_verbose:
        print(f"My model roc_auc: {my_roc_auc:.7}")
        print(f"Baseline model roc_auc: {baseline_roc_auc:.7}\n")
        print(f"Score limit for my model: {limit:.7}")
        print(f"Limit is passed: {my_roc_auc >= limit}")

    return my_roc_auc, baseline_roc_auc

Так как в стандартной реализации sklearn нет разделяющего правила misclassification error, будем сравнивать на дефолтном вероятностном правиле - критерий Джини.

In [8]:
from sklearn.tree import DecisionTreeClassifier

_ = compare_models(
    MyCoolDecisionTree(max_depth=6),
    DecisionTreeClassifier(max_depth=6), # Gini use for default
    X_train, y_train, X_test, y_test)

My model roc_auc: 0.9798308
Baseline model roc_auc: 0.9848214

Score limit for my model: 0.7878571
Limit is passed: True


C:\Users\Maksim Tseshnaty\AppData\Local\Temp\ipykernel_8028\3023303685.py:119: RuntimeWarning: invalid value encountered in scalar multiply
  return ((X_l.shape[0] / X.shape[0]) * self._H(X_l, y_l) +


**Задание 2. Модификация алгоритма для предсказания вероятностей классов (1б)**

Поддержите критейри Джини для построения дерева. Сравните результаты работы этих двух критерием на задаче классификации iris.


In [9]:
from typing import Literal


class MyCoolDecisionTreeClassifier(MyCoolDecisionTree):
    def __init__(self, 
                 max_depth: int | None = None, 
                 min_samples_leaf: int | None = None, 
                 min_samples_split: int | None = None,
                 criterion: Literal["gini", "misclserr"] = "gini"):
        super().__init__(
            max_depth, 
            min_samples_leaf, 
            min_samples_split)
        
        self.criterion = criterion
    
    def _H(self, X, y):
        if y.shape[0] == 0:
            return np.float32("+inf")
        
        cnts = np.unique_counts(y)
        if self.criterion == "misclserr":
            k_count = cnts.counts.max()
            return 1. - k_count / X.shape[0]
        
        elif self.criterion == "gini":
            p_ks = cnts.counts / y.shape[0]
            return 1 - np.sum(p_ks ** 2)
        
        else:
            raise ValueError(f"criterion: {self.criterion} is invalid!")

    

In [10]:
_ = compare_models(
    MyCoolDecisionTreeClassifier(max_depth=6, criterion="gini"),
    DecisionTreeClassifier(max_depth=6, criterion="gini", splitter="best"),
    X_train, y_train, X_test, y_test
)

My model roc_auc: 0.9841976
Baseline model roc_auc: 0.9848214

Score limit for my model: 0.7878571
Limit is passed: True


C:\Users\Maksim Tseshnaty\AppData\Local\Temp\ipykernel_8028\3023303685.py:119: RuntimeWarning: invalid value encountered in scalar multiply
  return ((X_l.shape[0] / X.shape[0]) * self._H(X_l, y_l) +


Можно заметить, что моя реализация алгоритма на тестовой выборке показывает результаты не хуже, а в случае тмерики $AUC-ROC$ даже лучше.

**Задание 3. Использование динамического программирования для ускорения алгоритма (2.5б)**

Используйте динамическое программирование, которое описано в главе Динамическое программирование учебника. Ваша сложность должна быть в этом месте должна быть $\mathcal{O}(hND\log(N))$. Сравните результаты работы наивной реализации дерева решений с оптимизированной, покажите, как растёт скорость работы, замерьте качество двух моделей.

In [11]:
class MyCoolDecisionTreeClassifierOptimized(MyCoolDecisionTreeClassifier):
    def __init__(self, 
                 max_depth = None, 
                 min_samples_leaf = None, 
                 min_samples_split = None, 
                 criterion = "gini"):
        super().__init__(
            max_depth, 
            min_samples_leaf, 
            min_samples_split, 
            criterion)
        
        self._precount = None
    


    def _get_split(self, X, y):
        min_loss = float("+inf")
        optimal_border = None

        for j in range(X.shape[1]):
            sorted_idx = np.argsort(X[:, j])
            sorted_x = X[sorted_idx, j]
            self._precount = None

            for i in range(X.shape[0] - 1):
                loss = self._get_criterion(y, sorted_idx, i)

                if np.abs(sorted_x[i] - sorted_x[i + 1]) < 1e-12:
                    continue
                if loss < min_loss:
                    min_loss = loss
                    threshold = (sorted_x[i] + sorted_x[i + 1]) / 2.0
                    optimal_border = (j, threshold)

        return min_loss, optimal_border
    


    def _get_criterion(self, y, sorted_idx, index):
        if index == y.shape[0] - 1:
            return np.float32("+inf")

        if self._precount is None:
            class_to_index = {c: i for i, c in enumerate(self._classes)}
            vals, counts = np.unique(y, return_counts=True)

            right_counts = np.zeros(len(self._classes), dtype=int)
            for v, c in zip(vals, counts):
                right_counts[class_to_index[v]] = c

            self._precount = {
                "left_counts": np.zeros(len(self._classes), dtype=int),
                "right_counts": right_counts,
                "left_max_count": 0,
                "right_maxes": self._get_right_maxes(y, sorted_idx),
                "class_to_index": class_to_index,
            }

        cls = y[sorted_idx[index]]
        cls_idx = self._precount["class_to_index"][cls]

        self._precount["left_counts"][cls_idx] += 1
        self._precount["right_counts"][cls_idx] -= 1

        self._precount["left_max_count"] = max(
            self._precount["left_max_count"],
            self._precount["left_counts"][cls_idx],
        )

        right_max_count = int(self._precount["right_maxes"][index + 1])

        return self._H(index, y.shape[0], right_max_count)


    def _H(self, index, n_samples, right_max_count):
        left_n = index + 1
        right_n = n_samples - left_n

        if self.criterion == "misclserr":
            left_err = 1.0 - self._precount["left_max_count"] / left_n
            right_err = 1.0 - right_max_count / right_n
            return ((left_n / n_samples) * left_err + 
                    (right_n / n_samples) * right_err)

        elif self.criterion == "gini":
            left_p = self._precount["left_counts"] / left_n
            right_p = self._precount["right_counts"] / right_n
            return ((left_n / n_samples) * (1.0 - np.sum(left_p ** 2)) + 
                    (right_n / n_samples) * (1.0 - np.sum(right_p ** 2)))

        else:
            raise ValueError(f"criterion {self.criterion} is invalid")

    
    def _get_right_maxes(self, y, sorted_idx):
        result = np.zeros_like(sorted_idx)
        classes_counts = {}
        cur_max = np.float32("-inf")
        for i in range(sorted_idx.shape[0] - 1, -1, -1):
            cur_cls = y[sorted_idx[i]]
            if cur_cls in classes_counts:
                classes_counts[cur_cls] += 1
            else:
                classes_counts[cur_cls] = 1
            
            cur_max = max(cur_max, classes_counts[cur_cls])
            result[i] = cur_max

        return np.array(result)


In [12]:
_ = compare_models(
    MyCoolDecisionTreeClassifierOptimized(max_depth=6, criterion="gini"),
    DecisionTreeClassifier(max_depth=6),
    X_train, y_train, X_test, y_test
)

My model roc_auc: 0.9841976
Baseline model roc_auc: 1.0

Score limit for my model: 0.8
Limit is passed: True


Как видно из сравнения выше, __MyCoolDecisionTreeClassifierOptimized__ выдаёт такие же результаты, укладывающиеся в допустимую ошибку.

Сравним время работы __MyCoolDecisionTreeClassifierOptimized__ и __MyCoolDecisionTreeClassifier__.  
Так как датасет iris довольно маленький для такого сравнения, я просто увеличу его до $100000$ объектов простым копированием и перемещиванием.

In [13]:
import time
np.random.seed = 42

idx = np.random.choice(X.shape[0], size=X.shape[0] * 70, replace=True)
X_aug = np.array(X)[idx]
y_aug = np.array(y)[idx]

In [14]:
X_aug.shape

(10500, 4)

In [15]:
def time_test(model):
    start = time.time()
    model.fit(X_aug, y_aug)
    fit_time = time.time() - start

    print(f"Roc_auc score: {roc_auc_score(y_aug, model.predict_proba(X_aug), multi_class='ovr')}")
    print(f"Fit time: {fit_time}")

In [16]:
time_test(MyCoolDecisionTreeClassifierOptimized(max_depth=5))

Roc_auc score: 0.999843682122001
Fit time: 1.357191562652588


In [17]:
time_test(MyCoolDecisionTreeClassifier(max_depth=5))

C:\Users\Maksim Tseshnaty\AppData\Local\Temp\ipykernel_8028\3023303685.py:119: RuntimeWarning: invalid value encountered in scalar multiply
  return ((X_l.shape[0] / X.shape[0]) * self._H(X_l, y_l) +


Roc_auc score: 0.999843682122001
Fit time: 24.852348804473877


Оптимзированный алгоритм работает намного быстрее, чем обычная наивная реализация через полный перебор.


### enhanced decision tree (4.5 балла)

Для тестирования решение понадобится датасет с категориальными признаками. Датасет с описанием расположен [здесь](http://archive.ics.uci.edu/dataset/45/heart+disease).

Код рассчитан на то, что вы их распакуете в директорию `datasets` в рабочей директории вашего ноутбука.

**Задание 4. Подготовка. (0.5 балла)**

Скачайте датасет, выкините или обработайте в нём категориальные признаки и заполните пропуски константой. Запустите вашу реализацию из секции `plain decision tree` и замерьте ваш результат классификации.

In [18]:
import numpy as np
import pandas as pd

names=[
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

num_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
cat_feaures = ["exang", "restecg", "fbs", "cp", "sex", "thal", "slope", "ca"]

df_clev = pd.read_csv("lab_1.4/heart+disease/processed.cleveland.data", header=None, names=names)
df_hung = pd.read_csv("lab_1.4/heart+disease/processed.hungarian.data", header=None, names=names)
df_swit = pd.read_csv("lab_1.4/heart+disease/processed.switzerland.data", header=None, names=names)
df_va   = pd.read_csv("lab_1.4/heart+disease/processed.va.data", header=None, names=names)

heart_disease = pd.concat([
    df_clev.replace(to_replace="?", value=np.nan).astype(float),
    df_hung.replace(to_replace="?", value=np.nan).astype(float),
    df_swit.replace(to_replace="?", value=np.nan).astype(float),
    df_va.replace(to_replace="?", value=np.nan).astype(float),
])

X, y = heart_disease.drop(["target"], axis=1), (heart_disease["target"] > 0).astype(float)

Мы бинаризировали наши метки, чтобы решать задачу бинарной классификации:

In [19]:
y.value_counts()

target
1.0    509
0.0    411
Name: count, dtype: int64

В данных очень много пропусков:

In [20]:
print("Процент пропусков в числовых фичах: ")
(X[num_features].isna().sum(axis=0) / X.shape[0] * 100)

Процент пропусков в числовых фичах: 


age         0.000000
trestbps    6.413043
chol        3.260870
thalach     5.978261
oldpeak     6.739130
dtype: float64

In [21]:
print("Процент пропусков в категориальных фичах: ")
(X[cat_feaures].isna().sum(axis=0) / X.shape[0] * 100)

Процент пропусков в категориальных фичах: 


exang       5.978261
restecg     0.217391
fbs         9.782609
cp          0.000000
sex         0.000000
thal       52.826087
slope      33.586957
ca         66.413043
dtype: float64

Обработаем пропуски в фичах:  
1. Числовые пропуски заполним средними.
2. Категориальные фичи с большим процентом пропусков уберём, а остальные заполним модой.

In [22]:
# features_to_drop = ['thal', 'slope', 'ca']
# X_nocat_nonans = X.drop(features_to_drop, axis=1)

# features_to_moda = ['exang', 'restecg', 'fbs']
# modas = X_nocat_nonans[features_to_moda].mode(axis=0)
# for feature in features_to_moda:
#     X_nocat_nonans.loc[X[feature].isna()] = modas[feature][0]

X_nocat_nonans = X.drop(cat_feaures, axis=1)

means = X_nocat_nonans[~X_nocat_nonans[num_features].isna()].mean(axis=0)
for value, feature in zip(means, means.index):
    X_nocat_nonans.loc[X_nocat_nonans[feature].isna(), feature] = value
    

In [23]:
X_nn_train, X_nn_test, y_nn_train, y_nn_test = train_test_split(np.array(X_nocat_nonans), np.array(y), test_size=0.33, random_state=42)

In [24]:
X_nocat_nonans.isna().any()

age         False
trestbps    False
chol        False
thalach     False
oldpeak     False
dtype: bool

In [25]:
_ = compare_models(
    MyCoolDecisionTreeClassifierOptimized(max_depth=6, criterion="gini"),
    DecisionTreeClassifier(max_depth=6, criterion="gini", splitter="best"),
    X_nn_train, y_nn_train, X_nn_test, y_nn_test
)

My model roc_auc: 0.789078
Baseline model roc_auc: 0.7794486

Score limit for my model: 0.6235589
Limit is passed: True


Как мы видим выше, просто удаление категориальных фичей и усреднение прпусков очень сильно ухудшает метрики.

**Задание 5. Поддержка категориальных признаков (2 балла)**

Поддержите категориальные фичи, как это описано в параграфе учебника про особенности данных. Сами категориальные фичи вы можете посмотреть в описании датасетта на сайте.

Сравните результаты работы реализации из секции `plain decision tree` без явной поддержки категориальных фичей, как вы делали при подготовке, с поддержкой категориальных. Покажите, как растёт или падает качество обученной модели на отложенной выборке.

In [26]:
class MyCoolDecisionTreeCatFeatBinClassifier(MyCoolDecisionTreeClassifierOptimized):
    def __init__(self, 
                 max_depth=None, 
                 min_samples_leaf=None, 
                 min_samples_split=None, 
                 criterion="gini",
                 cat_features_idx: list[int] = []):
        
        super().__init__(
            max_depth, 
            min_samples_leaf, 
            min_samples_split, 
            criterion)
        
        self.cat_features_idx = set(cat_features_idx)
        
    
    def _sort_by_feature(self, X, y, feature_idx) -> np.ndarray[int]:
        if feature_idx in self.cat_features_idx:
            X_1 = X[y == 1]

            cat_vals, _ = np.unique(X[:, feature_idx], return_counts=True)

            p_cat_vals = []
            for cat_val in cat_vals:
                cat_n_1 = X_1[X_1[:, feature_idx] == cat_val].shape[0]
                if cat_n_1 == 0:
                    p_cat_vals.append(0)
                else:
                    cat_n = X[X[:, feature_idx] == cat_val].shape[0]
                    p_cat_vals.append(cat_n_1 / cat_n)
            
            p_cat_vals = np.array(p_cat_vals)
            sorted_cat_vals = cat_vals[np.argsort(p_cat_vals)]

            cat_to_rank = {cat: i for i, cat in enumerate(sorted_cat_vals)}
            ranks = np.vectorize(cat_to_rank.get)(X[:, feature_idx])
            return np.argsort(ranks)

        return np.argsort(X[:, feature_idx])

    
    def _get_split(self, X, y):
        min_loss = float("+inf")
        optimal_border = None

        for j in range(X.shape[1]):
            sorted_idx = self._sort_by_feature(X, y, j)

            sorted_x = X[sorted_idx, j]
            self._precount = None

            for i in range(X.shape[0] - 1):
                loss = self._get_criterion(y, sorted_idx, i)

                if np.abs(sorted_x[i] - sorted_x[i + 1]) < 1e-10:
                    continue
                if loss < min_loss:
                    min_loss = loss
                    threshold = (sorted_x[i] + sorted_x[i + 1]) / 2.0
                    optimal_border = (j, threshold)

        return min_loss, optimal_border

In [27]:
features_to_drop = ['thal', 'slope', 'ca', 'exang', 'restecg', 'fbs']
X_non_nans = X.drop(features_to_drop, axis=1)

means = X_non_nans[~X_non_nans[num_features].isna()].mean(axis=0)
for value, feature in zip(means, means.index):
    X_non_nans.loc[X_non_nans[feature].isna(), feature] = value

In [28]:
X_non_nans_train, X_non_nans_test, y_non_nans_train, y_non_nans_test = train_test_split(np.array(X_non_nans), np.array(y), test_size=0.33, random_state=42)

In [29]:
_ = compare_models(
    MyCoolDecisionTreeCatFeatBinClassifier(max_depth=6, cat_features_idx=[1, 2]),
    DecisionTreeClassifier(max_depth=6),
    X_non_nans_train, y_non_nans_train, X_non_nans_test, y_non_nans_test
)

My model roc_auc: 0.8252429
Baseline model roc_auc: 0.8141406

Score limit for my model: 0.6513125
Limit is passed: True


Как можно видеть выше, моя модель показывает даже более лучший результат, чем байзлайн модель. Возможно это связано с том, что __sklearn.tree.DecisionTreeClassifier__ не обрабатывает категориальные фичи, так же, как я в своей моделе.  
Насколько мне известно он их просто сортирует по значению.

**Задание 6. Работа с пропусками (2 балла)**

Обработайте пропуски, как это описано в параграфе учебника про особенности данных.

Сравните результаты работы реализации из секции `plain decision tree` с заполнением пропусков, как вы делали при подготовке, с поддержкой пропусков в фичах. Покажите, как растёт или падает качество обученной модели на отложенной выборке.

In [30]:
from typing import Literal


class TreeNodeNan(TreeNode):
    def __init__(self, depth: int | None = None):
        super().__init__(depth)
        
        self.nan_to_go: Literal["left", "right"] | None = None

    def predict(self, x):
        if self.is_leaf:
            return self.answer

        assert (
            self.feature_idx is not None and self.value is not None
        ), "at prediction split values should be specified"

        f_val = x[self.feature_idx]
        if np.isnan(f_val):
            assert self.nan_to_go in ("left", "right"), "nan_to_go must be set for nodes with NaNs"
            if self.nan_to_go == "left":
                return self.left.predict(x)
            else:
                return self.right.predict(x)

        
        if f_val < self.value:
            return self.left.predict(x)
        else:
            return self.right.predict(x)




class MyCoolDecisionTreeCatNanFeatBinClassifier(MyCoolDecisionTreeCatFeatBinClassifier):
    def __init__(self, 
                 max_depth=None, 
                 min_samples_leaf=None, 
                 min_samples_split=None, 
                 criterion="gini", 
                 cat_features_idx = []):
        
        super().__init__(
            max_depth, 
            min_samples_leaf, 
            min_samples_split, 
            criterion, 
            cat_features_idx)
    
    
    def fit(self, X, y):
        valid_X = X

        if self.cat_features_idx and np.any(np.isnan(X[:, list(self.cat_features_idx)])):
            
            for feature_idx in self.cat_features_idx:
                if isinstance(valid_X[0, feature_idx], str):
                    valid_X[:, feature_idx] = "-1"
                else:
                    valid_X[:, feature_idx] = -1.

        self._classes = np.unique(y)
        self.root = TreeNodeNan(depth=1)
        self._build_tree(self.root, valid_X, y)
        return self


    def _build_tree(self, node: TreeNodeNan, X, y) -> TreeNodeNan:
        if self._stop_criterion(node, X):
            node.answer = self._get_answer(X, y)
            return node

        _, optimal_border, optimal_nan_to_go = self._get_split(X, y)
        if optimal_border is None:
            node.answer = self._get_answer(X, y)
            return node

        feature_idx, t = optimal_border
        node.feature_idx = feature_idx
        node.value = t
        node.nan_to_go = optimal_nan_to_go

        nan_mask = np.isnan(X[:, feature_idx])
        non_nan_mask = ~nan_mask

        mask_non_nan_left = (X[:, feature_idx] < t) & non_nan_mask

        if node.nan_to_go == "left":
            left_mask = mask_non_nan_left | nan_mask
        elif node.nan_to_go == "right":
            left_mask = mask_non_nan_left
        else:
            left_mask = (X[:, feature_idx] < t)

        X_l, y_l = X[left_mask], y[left_mask]
        X_r, y_r = X[~left_mask], y[~left_mask]

        left = TreeNodeNan(node.depth + 1)
        right = TreeNodeNan(node.depth + 1)
        node.left = self._build_tree(left, X_l, y_l)
        node.right = self._build_tree(right, X_r, y_r)

        return node



    def _get_cat_feature_split(self, X, y, f_idx) -> tuple[float, float]:
        min_loss = float("+inf")
        optimal_border = None
        sorted_idx = self._sort_by_feature(X, y, f_idx)

        sorted_x = X[sorted_idx, f_idx]
        self._precount = None

        for i in range(X.shape[0] - 1):
            loss = self._get_criterion(y, sorted_idx, i)

            if np.abs(sorted_x[i] - sorted_x[i + 1]) < 1e-3:
                continue
            if loss < min_loss:
                min_loss = loss
                threshold = (sorted_x[i] + sorted_x[i + 1]) / 2.0
                optimal_border = (f_idx, threshold)

        return min_loss, optimal_border

    
    def _get_num_feature_split(self, X, y, f_idx):
        prep = self._prepare_numeric_feature(X, y, f_idx)
        if prep is None:
            return float("+inf"), None, None

        (
            sorted_idx,
            sorted_x,
            nan_counts,
            class_to_index,
            n_total
        ) = prep

        counters = self._init_numeric_counters(y, sorted_idx, class_to_index)

        return self._eval_numeric_split(
            y=y,
            f_idx=f_idx,
            sorted_idx=sorted_idx,
            sorted_x=sorted_x,
            nan_counts=nan_counts,
            class_to_index=class_to_index,
            counters=counters,
            n_total=n_total
        )



    def _prepare_numeric_feature(self, X, y, f_idx):
        xf = X[:, f_idx]
        nan_mask = np.isnan(xf)
        non_nan_mask = ~nan_mask

        if non_nan_mask.sum() <= 1:
            return None

        global_idx = np.where(non_nan_mask)[0]
        local_sorted_idx = np.argsort(xf[non_nan_mask])
        sorted_idx = global_idx[local_sorted_idx]
        sorted_x = xf[sorted_idx]

        class_to_index = {c: i for i, c in enumerate(self._classes)}

        nan_counts = np.zeros(len(self._classes), dtype=int)
        for lbl in y[nan_mask]:
            nan_counts[class_to_index[lbl]] += 1

        n_total = len(sorted_idx) + nan_counts.sum()

        return sorted_idx, sorted_x, nan_counts, class_to_index, n_total



    def _init_numeric_counters(self, y, sorted_idx, class_to_index):
        right_counts = np.zeros(len(self._classes), dtype=int)
        for lbl in y[sorted_idx]:
            right_counts[class_to_index[lbl]] += 1

        left_counts = np.zeros(len(self._classes), dtype=int)

        return {
            "left_counts": left_counts,
            "right_counts": right_counts
        }




    def _eval_numeric_split(
        self,
        y,
        f_idx,
        sorted_idx,
        sorted_x,
        nan_counts,
        class_to_index,
        counters,
        n_total
    ):
        min_loss = float("+inf")
        best_border = None
        best_nan_go = None

        left_counts = counters["left_counts"]
        right_counts = counters["right_counts"]

        for i in range(len(sorted_idx) - 1):
            cls = y[sorted_idx[i]]
            cls_idx = class_to_index[cls]

            left_counts[cls_idx] += 1
            right_counts[cls_idx] -= 1

            if np.abs(sorted_x[i] - sorted_x[i + 1]) < 1e-12:
                continue

            threshold = (sorted_x[i] + sorted_x[i + 1]) / 2.0

            loss_l = self._loss_nan_left(
                left_counts, right_counts, nan_counts, i + 1, n_total
            )
            loss_r = self._loss_nan_right(
                left_counts, right_counts, nan_counts, i + 1, n_total
            )

            if loss_l < min_loss:
                min_loss = loss_l
                best_border = (f_idx, threshold)
                best_nan_go = "left"

            if loss_r < min_loss:
                min_loss = loss_r
                best_border = (f_idx, threshold)
                best_nan_go = "right"

        return min_loss, best_border, best_nan_go




    def _loss_nan_left(self, left_counts, right_counts, nan_counts, left_n, n_total):
        left_counts_all = left_counts + nan_counts
        right_n = n_total - left_n - nan_counts.sum()

        if left_n + nan_counts.sum() < self.min_samples_leaf or right_n < self.min_samples_leaf:
            return float("+inf")

        return self._weighted_impurity(
            left_counts_all,
            right_counts,
            left_n + nan_counts.sum(),
            right_n,
            n_total
        )



    def _loss_nan_right(self, left_counts, right_counts, nan_counts, left_n, n_total):
        right_counts_all = right_counts + nan_counts
        right_n = n_total - left_n

        if left_n < self.min_samples_leaf or right_n < self.min_samples_leaf:
            return float("+inf")

        return self._weighted_impurity(
            left_counts,
            right_counts_all,
            left_n,
            right_n,
            n_total
        )



    def _weighted_impurity(
        self,
        left_counts,
        right_counts,
        left_n,
        right_n,
        n_total
    ):
        if self.criterion == "misclserr":
            left_err = 1.0 - left_counts.max() / left_n
            right_err = 1.0 - right_counts.max() / right_n

        elif self.criterion == "gini":
            left_p = left_counts / left_n
            right_p = right_counts / right_n
            left_err = 1.0 - np.sum(left_p ** 2)
            right_err = 1.0 - np.sum(right_p ** 2)

        else:
            raise ValueError(self.criterion)

        return (
            (left_n / n_total) * left_err +
            (right_n / n_total) * right_err
        )




    def _get_split(self, X, y):
        min_loss = float("+inf")
        optimal_border = None
        optimal_nan_to_go = None

        for j in range(X.shape[1]):
            if j in self.cat_features_idx:
                loss, border = self._get_cat_feature_split(X, y, j)
                nan_to_go = None
            else:
                loss, border, nan_to_go = self._get_num_feature_split(X, y, j)

            if border is None:
                continue

            if loss < min_loss:
                min_loss = loss
                optimal_border = border
                optimal_nan_to_go = nan_to_go

        return min_loss, optimal_border, optimal_nan_to_go

        

In [31]:
cat_features_idx = [X.columns.get_loc(cat) for cat in cat_feaures]
X_train, X_test, y_train, y_test = train_test_split(np.array(X), np.array(y), test_size=0.33, random_state=42)

In [32]:
_ = compare_models(
    MyCoolDecisionTreeCatNanFeatBinClassifier(max_depth=6, cat_features_idx=cat_features_idx),
    DecisionTreeClassifier(max_depth=6),
    X_train, y_train, X_test, y_test
)

My model roc_auc: 0.7843512
Baseline model roc_auc: 0.7654223

Score limit for my model: 0.6123379
Limit is passed: True


Как можно увидеть $ROC-AUC$ моей реализации выше, так как в ней помимо реализации обработки пропусков ещё добавлена обработка категориальных фичей (правда работать она полностью корретно будет только для критерия Джини)

## random forest (4 балла)

Теорию для данного задания вы можете найти в соответсвующей [главе учебника](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii).

В данной разделе вы можете использовать как вашу реализацию из предыдущего раздела, так и реализацию дерева решений из sklearn. При использовании реализации sklearn вы получите не более половины баллов за задание, когда использование собственного дерева решений может вам принести полный балл.

Интерфейс можете подсмотреть в [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

random forest classifier должен поддерживать следующие параметры:
- максимальное количество деревьев в лесе (*n_estimators*)
- максимальное количество семплов для обучения одного дерева (*max_samples*)
- максимальная глубина дерева (*max_depth*)


**Задание 7. Беггинг над решающими деревьями (1 балла)**

Используйте беггинг над решающими деревьями. Помните, что беггинг призван уменьшить variance, но не bias. Чтобы ваше смещение было небольшим, берите достаточно глубокие деревья.

Сравните результаты работы реализации из секции `decision tree` с беггингом деревьев решений. Переберите размер выборки при бутстрапе, при котором у вашего беггинга будет максимальный скор. Удалось ли добиться роста метрики относительно базового алгоритма? Почему?

In [ ]:
from typing import Literal
from sklearn.base import BaseEstimator, ClassifierMixin


class MyCorrRandomForestBinClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self,
                n_estimators: int,
                max_samples: int = None,
                max_depth = None, 
                min_samples_leaf: int = 1, 
                min_samples_split: int = 2, 
                criterion: Literal["gini", "misce"] = "gini", 
                cat_features_idx: list[int] = []):
        
        assert n_estimators, "you nedd to specify n_estimators"
        
        self.n_estimators = n_estimators
        self.features_idx = []
        self.estimators = []
        self.max_samples = max_samples

        self.criterion = criterion
        self.min_sample_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.cat_features_idx = np.array(cat_features_idx)
        self.max_depth = max_depth


    def fit(self, X: np.ndarray, y: np.ndarray):
        self.max_samples = self.max_samples or X.shape[0]
        for _ in range(self.n_estimators):
            sample_idx = np.random.choice(X.shape[0], size=self.max_samples, replace=True)

            X_tree_train = X[sample_idx]
            y_tree_train = y[sample_idx]

            tree = MyCoolDecisionTreeCatNanFeatBinClassifier(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_sample_leaf,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                cat_features_idx=self.cat_features_idx
            )
            tree.fit(X_tree_train, y_tree_train)
            self.estimators.append(tree)


    def predict_proba(self, X: np.ndarray):
        return np.mean([tree.predict_proba(X) for tree in self.estimators], axis=0)

In [34]:
import tqdm


def test_forest(
        n_estimators: int,
        max_depth: int,
        decisionTreeClassifier,
        randomForestClassifier,
        is_verbose: bool = False
) -> tuple[np.ndarray[float], np.ndarray[float]]:

    n_train = X_train.shape[0]
    max_samples = [
        np.int32(n_train * 0.5), 
        n_train, 
        np.int32(n_train * 1.5), 
        np.int32(n_train * 2)
    ]
    ans_score = []
    tree_score = []

    for max_sample in tqdm.tqdm(max_samples):
        if is_verbose:
            print(f"====== {max_sample} samples ======")
        my_score, tr_score = compare_models(
            randomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, cat_features_idx=cat_features_idx),
            decisionTreeClassifier(max_depth=max_depth),
            X_train, y_train, X_test, y_test,
            is_verbose=is_verbose
        )
        if is_verbose:
            print("===========================")
        ans_score.append(my_score)
        tree_score.append(tr_score)


    if is_verbose:
        print(f"\nThe best ans score: {max(ans_score):.7f}")
        print(f"The best tree score: {max(tree_score):.7f}")
    
    return np.array(ans_score), np.array(tree_score)

Протестируем наше дерево в сравнении с реализацией дерева sklean.tree.DecisionTreeClassifier и моей реализацией дерева.  
Поскольку лес у нас случайный, сделаем несколько замеров скора на каждом значений __max_samples__ и усредним.

In [35]:
def mean_random_forest_test(
        n_estimators=30,
        max_depth=10,
        randomForestClassifier=MyCorrRandomForestBinClassifier,
        decisionTreeClassifier=DecisionTreeClassifier,
        is_verbose=False
):

    ans_scores = []
    tree_scores = []
    for _ in range(10):
        ans_score, tree_score = test_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            randomForestClassifier=randomForestClassifier,
            decisionTreeClassifier=decisionTreeClassifier,
            is_verbose=is_verbose
        )

        ans_scores.append(ans_score)
        tree_scores.append(tree_score)

    ans_scores = np.array(ans_scores)
    tree_scores = np.array(tree_scores)

    ans_score_mean = ans_scores.mean(axis=0)
    tree_score_mean = tree_scores.mean(axis=0)
    
    return ans_score_mean, tree_score_mean

In [36]:
ans_score_mean, tree_score_mean = mean_random_forest_test(
    n_estimators=20,
    max_depth=9,
    randomForestClassifier=MyCorrRandomForestBinClassifier,
    decisionTreeClassifier=DecisionTreeClassifier
)

print(ans_score_mean, tree_score_mean)
print(f"Max ans mean score: {max(ans_score_mean)}\nMax tree mean score: {max(tree_score_mean)}")

100%|██████████| 4/4 [00:46<00:00, 11.55s/it]

[0.82003693 0.82045465 0.82391725 0.82079761] [0.70493119 0.70644594 0.70806622 0.7157455 ]
Max ans mean score: 0.8239172492635095
Max tree mean score: 0.7157455041111551


Как мы видим ансамбль намного лучше справляется по скору, чем одно дерево решений.  
Связано это с тем, что ансамбль уменьшачет дисперсию пердсказания, а значит увеличивает итоговый скор.

**Задание 8. Feature subsampling (3 балла)**

Во время беггинга вводится предположение, что базовые алгоритмы некоррелированы. На самом деле это не так, потому что это одна модель, обучающаяся на данных из одного распределения. Чтобы разломать корреляцию используют семплирование фичей при обучении нового алгоритма: выбирают случайное подмножество фичей, на котором будет обучаться очередная модель.

Используйте feature subsampling во время беггинга:
1. фиксируйте набор фичей для построения дерева перед началом построения (1б)
2. выбирайте случайно множество фичей при выборе разделяющего правила во внутренней ноде (1б)

Сравните результаты работы реализации из секции `decision tree` c деревьями решений (1б). В качестве количества семплированных фичей возьмите квадратный корень из их общего количества, как было предложено в учебнике. Например, если у вас 25 исходных фичей, вы будете выбирать по 5 фичей для обучения очередной модели при беггинге. Переберите размер выборки при бутстрапе, при котором у вашего беггинга будет максимальный скор. Удалось ли добиться роста метрики относительно базового алгоритма и беггинга без feature subsampling? Почему?

Поскольку случайно выбирать подмножество фичей исходного количества фичей или больше смысла не имеет, я решил, что буду выбирать $2/3$ фичей случайно для каждого разделения.

In [37]:
from typing import Literal


class MyCoolRandomDecisionTreeCatNanFeatBinClassifier(
    MyCoolDecisionTreeCatNanFeatBinClassifier):
    def __init__(self, 
                 max_depth=None, 
                 min_samples_leaf=None, 
                 min_samples_split=None, 
                 criterion="gini", 
                 cat_features_idx=[]):
        
        super().__init__(
            max_depth, 
            min_samples_leaf, 
            min_samples_split, 
            criterion, 
            cat_features_idx)

    def _get_random_reatures(self, X):
        features_n = np.int32(X.shape[1] * 2 / 3)
        features_split_idx = np.random.choice(X.shape[1], size=features_n, replace=False)

        X_to_split = X[:, features_split_idx]
        split_cat_features = np.where(np.isin(features_split_idx, self.cat_features_idx))[0].tolist()

        return X_to_split, split_cat_features, features_split_idx



    def _build_tree(self, node: TreeNodeNan, X, y) -> TreeNodeNan:
        if self._stop_criterion(node, X):
            node.answer = self._get_answer(X, y)
            return node
        
        X_to_split, split_cat_features_idx, features_split_idx = self._get_random_reatures(X)
        

        _, optimal_border, optimal_nan_to_go = self._get_split(X_to_split, y, split_cat_features_idx)
        if optimal_border is None:
            node.answer = self._get_answer(X, y)
            return node

        feature_idx_split_local, t = optimal_border
        feature_idx = features_split_idx[feature_idx_split_local]
        node.feature_idx = feature_idx
        node.value = t
        node.nan_to_go = optimal_nan_to_go

        nan_mask = np.isnan(X[:, feature_idx])
        non_nan_mask = ~nan_mask

        mask_non_nan_left = (X[:, feature_idx] < t) & non_nan_mask

        if node.nan_to_go == "left":
            left_mask = mask_non_nan_left | nan_mask
        elif node.nan_to_go == "right":
            left_mask = mask_non_nan_left
        else:
            left_mask = (X[:, feature_idx] < t)

        X_l, y_l = X[left_mask], y[left_mask]
        X_r, y_r = X[~left_mask], y[~left_mask]

        left = TreeNodeNan(node.depth + 1)
        right = TreeNodeNan(node.depth + 1)
        node.left = self._build_tree(left, X_l, y_l)
        node.right = self._build_tree(right, X_r, y_r)

        return node
    

    def _get_split(self, X, y, cat_features_idx):
        min_loss = float("+inf")
        optimal_border = None
        optimal_nan_to_go = None

        for j in range(X.shape[1]):
            if j in cat_features_idx:
                loss, border = self._get_cat_feature_split(X, y, j)
                nan_to_go = None
            else:
                loss, border, nan_to_go = self._get_num_feature_split(X, y, j)

            if border is None:
                continue

            if loss < min_loss:
                min_loss = loss
                optimal_border = border
                optimal_nan_to_go = nan_to_go

        return min_loss, optimal_border, optimal_nan_to_go



class MyRandom2ForestBinClassifier:
    def __init__(self,
                n_estimators: int,
                max_samples: int = None,
                max_depth = None, 
                min_samples_leaf: int = 1, 
                min_samples_split: int = 2, 
                criterion: Literal["gini", "misce"] = "gini", 
                cat_features_idx: list[int] = []):
        
        assert n_estimators, "you nedd to specify n_estimators"
        
        self.n_estimators = n_estimators
        self.features_idx = []
        self.estimators = []
        self.max_samples = max_samples

        self.criterion = criterion
        self.min_sample_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.cat_features_idx = np.array(cat_features_idx)
        self.max_depth = max_depth


    def fit(self, X: np.ndarray, y: np.ndarray):
        self.max_samples = self.max_samples or X.shape[0]
        n_features = int(np.sqrt(X.shape[1]))

        for _ in range(self.n_estimators):
            features_idx = np.random.choice(X.shape[1], size=n_features, replace=False) # fix random features
            sample_idx = np.random.choice(X.shape[0], size=self.max_samples, replace=True)

            tree_cat_features = np.where(np.isin(features_idx, self.cat_features_idx))[0].tolist()

            self.features_idx.append(features_idx)

            X_tree_train = X[sample_idx]
            X_tree_train = X_tree_train[:, features_idx]
            y_tree_train = y[sample_idx]

            tree = MyCoolRandomDecisionTreeCatNanFeatBinClassifier(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_sample_leaf,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                cat_features_idx=tree_cat_features
            )
            tree.fit(X_tree_train, y_tree_train)
            self.estimators.append(tree)


    def predict_proba(self, X: np.ndarray):
        return np.mean([tree.predict_proba(X[:, features_idx]) for tree, features_idx in zip(self.estimators, self.features_idx)], axis=0)

In [38]:
ans_score_mean, tree_score_mean = mean_random_forest_test(
    n_estimators=20,
    max_depth=9,
    randomForestClassifier=MyRandom2ForestBinClassifier,
    decisionTreeClassifier=DecisionTreeClassifier
)

print(ans_score_mean, tree_score_mean)
print(f"Max ans mean score: {max(ans_score_mean)}\nMax tree mean score: {max(tree_score_mean)}")

100%|██████████| 4/4 [00:01<00:00,  2.64it/s]

[0.78851515 0.79732225 0.7926329  0.8044761 ] [0.70597766 0.70467397 0.70528514 0.7059271 ]
Max ans mean score: 0.8044761025370443
Max tree mean score: 0.705977663456888


Несмотря на мою доработки в рандомном лесу и рандомном дереве решений:
1. Рандомная фиксация всех фич для каждого дерева.
2. Рандомный выбор подмножества фич в каждом узле.  

Скор моего решения всё равно выше, чем у простого дерева, но при этом меньше, чем у реализации леса с простым беггингом. Я думаю это связано с тем, что в тестируемом датасете всего 13 фичей, np.sqrt(13) ~ 3.6, то есть у каждого дерева уже будет 3 фичи, а если в каждом узле ещё рандомно выбирать 2/3 фичей, то поиск лучшего сплита будет вообще идти примерно по 1 фиче, что не есть хорошо, мы упускаем слишком много вариантов более хорошего сплита.